# Local Language Translator using AI
Fine-tune NLLB-200 on your own sentence pairs.

**Before running:** Runtime > Change runtime type > T4 GPU. Upload `data.csv` (columns: `src`, `tgt`) to `MyDrive/translator/`.

## Step 1: Check GPU

In [ ]:
!nvidia-smi

## Step 2: Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/translator

## Step 3: Install libraries

In [ ]:
!pip install -q transformers datasets sacrebleu sentencepiece accelerate gradio

## Step 4: Load and clean data

In [ ]:
import pandas as pd
from datasets import Dataset

df = pd.read_csv('/content/drive/MyDrive/translator/data.csv')
df = df.dropna().drop_duplicates()
print(len(df))
df.head()

## Step 5: Load model and prepare data
Change `SRC` to your language's NLLB code (example: `asm_Beng`).

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

name = 'facebook/nllb-200-distilled-600M'
SRC, TGT = 'asm_Beng', 'eng_Latn'   # change SRC to your language code

tok = AutoTokenizer.from_pretrained(name, src_lang=SRC, tgt_lang=TGT)
model = AutoModelForSeq2SeqLM.from_pretrained(name)

ds = Dataset.from_pandas(df).train_test_split(test_size=0.1, seed=42)

def prep(b):
    return tok(b['src'], text_target=b['tgt'], max_length=128, truncation=True)

tokenized = ds.map(prep, batched=True, remove_columns=ds['train'].column_names)

## Step 6: Train and save

In [ ]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

args = Seq2SeqTrainingArguments(
    output_dir='/content/drive/MyDrive/translator/checkpoints',
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    num_train_epochs=3,
    fp16=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    data_collator=DataCollatorForSeq2Seq(tok, model=model),
)
trainer.train()

trainer.save_model('/content/drive/MyDrive/translator/final')
tok.save_pretrained('/content/drive/MyDrive/translator/final')

## Step 7: Test the translator

In [ ]:
def translate(text):
    inputs = tok(text, return_tensors='pt').to(model.device)
    out = model.generate(
        **inputs,
        forced_bos_token_id=tok.convert_tokens_to_ids(TGT),
        max_new_tokens=128,
    )
    return tok.decode(out[0], skip_special_tokens=True)

print(translate('your sentence here'))

## Step 8: Evaluate (BLEU and chrF)

In [ ]:
import sacrebleu

test = ds['test']
preds = [translate(s) for s in test['src']]
refs = [list(test['tgt'])]
print('BLEU:', sacrebleu.corpus_bleu(preds, refs).score)
print('chrF:', sacrebleu.corpus_chrf(preds, refs).score)

## Step 9: Demo with a shareable link

In [ ]:
import gradio as gr

gr.Interface(translate, 'text', 'text', title='My Translator').launch(share=True)